In [1]:
#!pip install crewai crewai_tools langchain langchain_community langchain_ollama streamlit duckduckgo-search

In [2]:
from crewai import LLM
from langchain_ollama.llms import OllamaLLM
from langchain_groq.chat_models import ChatGroq

llm = LLM(
    model="ollama/llama3.2",
    base_url="http://localhost:11434"
)

#llm = LLM(model="groq/openai/gpt-oss-20b")


In [3]:
from crewai.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults 
import json


@tool
def search_web_tool(query: str):
    """
    Searches the web and returns results.
    """
    search_tool = DuckDuckGoSearchResults(num_results=5, verbose=True)
    return search_tool.run(query)


In [4]:
from crewai import Agent

# Agent Resercher
guide_expert= Agent( 
    role="City Local Guide Expert",
    goal="Provides information on things to do in the city based on the user's interests.",
    backstory="""A local expert with a passion for sharing the best experiences and hidden gems of their city.""",
    tools=[search_web_tool],
    verbose=True,
    max_iter=2,
    llm=llm,  #ChatOpenAI(temperature=0, model="gpt-4o-mini"),
    allow_delegation=False,
)

In [5]:
# Agent city expert
location_expert = Agent(
    role="Travel Trip Expert",
    goal="Adapt to the user destination vity language (French if city in French Country. Gather helpful information about to the city and city during travel.",
    backstory="""A seasoned traveler who has explored various destinations and knows the ins and outs of travel logistics.""",
    tools=[search_web_tool],
    verbose=True,
    max_iter=2,
    llm=llm,
    allow_delegation=False,
)


In [6]:
planner_expert = Agent(
    role="Travel Planning Expert",
    goal="Compiles all gathered information to provide a comprehensive travel plan.",
    backstory="""
    You are a professional guide with a passion for travel.
    An organizational wizard who can turn a list of possibilities into a seamless itinerary.
    """,
    tools=[search_web_tool],
    verbose=True,
    max_iter=2,
    llm=llm,
    allow_delegation=False,
)


In [7]:
from datetime import datetime
from crewai import Task

from_city = "India"
destination_city = "Rome"
date_from = "1st March 2025"
date_to = "7th March 2025"
interests = "sight seeing and good food"

location_task = Task(
    description=f"""
    In French : This task involves a comprehensive data collection process to provide the traveler with essential information about their destination. It includes researching and compiling details on various accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of living in the area. The task also covers transportation options, visa requirements, and any travel advisories that may be relevant.
    consider also the weather conditions forcast on the travel dates. and all the events that may be relevant to the traveler during the trip period.
    
    Traveling from : {from_city}
    Destination city : {destination_city}
    Arrival Date : {date_from}
    Departure Date : {date_to}

    Follow this rules : 
    1. if the {destination_city} is in a French country : Respond in FRENCH.
    """,
    expected_output=f"""
    if the {destination_city} is in a French country : Respond in FRENCH.
    In markdown format : A detailed markdown report that includes a curated list of recommended places to stay, a breakdown of daily living expenses, and practical travel tips to ensure a smooth journey.
    """,
    agent=location_expert,
    output_file='agentic-ai/crewai/city_report.md',
)





In [8]:
guide_task = Task(
    description=f"""
    if the {destination_city} is in a French country : Respond in FRENCH.
    Tailored to the traveler's personal {interests}, this task focuses on creating an engaging and informative guide to the city's attractions. It involves identifying cultural landmarks, historical spots, entertainment venues, dining experiences, and outdoor activities that align with the user's preferences such {interests}. The guide also highlights seasonal events and festivals that might be of interest during the traveler's visit.
    Destination city : {destination_city}
    interests : {interests}
    Arrival Date : {date_from}
    Departure Date : {date_to}

    Follow this rules : 
    1. if the {destination_city} is in a French country : Respond in FRENCH.
    """,
    expected_output=f"""
    An interactive markdown report that presents a personalized itinerary of activities and attractions, complete with descriptions, locations, and any necessary reservations or tickets.
    """,

    agent=guide_expert,
    output_file='agentic-ai/crewai/guide_report.md',
)


In [9]:
planner_task = Task(
    description=f"""
    This task synthesizes all collected information into a detaileds introduction to the city (description of city and presentation, in 3 pragraphes) cohesive and practical travel plan. and takes into account the traveler's schedule, preferences, and budget to draft a day-by-day itinerary. The planner also provides insights into the city's layout and transportation system to facilitate easy navigation.
    Destination city : {destination_city}
    interests : {interests}
    Arrival Date : {date_from}
    Departure Date : {date_to}

    Follow this rules : 
    1. if the {destination_city} is in a French country : Respond in FRENCH.
    """,
    expected_output="""
    if the {destination_city} is in a French country : Respond in FRENCH.
    A rich markdown document with emojis on each title and subtitle, that :
    In markdown format : 
    # Welcome to {destination_city} :
    A 4 paragraphes markdown formated including :
    - a curated articles of presentation of the city, 
    - a breakdown of daily living expenses, and spots to visit.
    # Here's your Travel Plan to {destination_city} :
    Outlines a daily detailed travel plan list with time allocations and details for each activity, along with an overview of the city's highlights based on the guide's recommendations
    """,
    context=[location_task, guide_task],
    #context=context,
    agent=planner_expert,
    output_file='agentic-ai/crewai/travel_plan.md',
)


In [10]:
# Task: Location
def location_task(agent, from_city, destination_city, date_from, date_to):
    return Task(
        description=f"""
        In French : This task involves a comprehensive data collection process to provide the traveler with essential information about their destination. It includes researching and compiling details on various accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of living in the area. The task also covers transportation options, visa requirements, and any travel advisories that may be relevant.
        consider also the weather conditions forcast on the travel dates. and all the events that may be relevant to the traveler during the trip period.
        
        Traveling from : {from_city}
        Destination city : {destination_city}
        Arrival Date : {date_from}
        Departure Date : {date_to}

        Follow this rules : 
        1. if the {destination_city} is in a French country : Respond in FRENCH.
        """,
        expected_output=f"""
        if the {destination_city} is in a French country : Respond in FRENCH.
        In markdown format : A detailed markdown report that includes a curated list of recommended places to stay, a breakdown of daily living expenses, and practical travel tips to ensure a smooth journey.
        """,
        agent=agent,
        output_file='agentic-ai/crewai/city_report.md',
    )

# Task: Location
def guide_task(agent, destination_city, interests, date_from, date_to):    
    return Task(
        description=f"""
        if the {destination_city} is in a French country : Respond in FRENCH.
        Tailored to the traveler's personal {interests}, this task focuses on creating an engaging and informative guide to the city's attractions. It involves identifying cultural landmarks, historical spots, entertainment venues, dining experiences, and outdoor activities that align with the user's preferences such {interests}. The guide also highlights seasonal events and festivals that might be of interest during the traveler's visit.
        Destination city : {destination_city}
        interests : {interests}
        Arrival Date : {date_from}
        Departure Date : {date_to}

        Follow this rules : 
        1. if the {destination_city} is in a French country : Respond in FRENCH.
        """,
        expected_output=f"""
        An interactive markdown report that presents a personalized itinerary of activities and attractions, complete with descriptions, locations, and any necessary reservations or tickets.
        """,

        agent=agent,
        output_file='agentic-ai/crewai/guide_report.md',
    )


# Task: Planner
def planner_task(context, agent, destination_city, interests, date_from, date_to):
    return Task(
        description=f"""
        This task synthesizes all collected information into a detaileds introduction to the city (description of city and presentation, in 3 pragraphes) cohesive and practical travel plan. and takes into account the traveler's schedule, preferences, and budget to draft a day-by-day itinerary. The planner also provides insights into the city's layout and transportation system to facilitate easy navigation.
        Destination city : {destination_city}
        interests : {interests}
        Arrival Date : {date_from}
        Departure Date : {date_to}

        Follow this rules : 
        1. if the {destination_city} is in a French country : Respond in FRENCH.
        """,
        expected_output="""
        if the {destination_city} is in a French country : Respond in FRENCH.
        A rich markdown document with emojis on each title and subtitle, that :
        In markdown format : 
        # Welcome to {destination_city} :
        A 4 paragraphes markdown formated including :
        - a curated articles of presentation of the city, 
        - a breakdown of daily living expenses, and spots to visit.
        # Here's your Travel Plan to {destination_city} :
        Outlines a daily detailed travel plan list with time allocations and details for each activity, along with an overview of the city's highlights based on the guide's recommendations
        """,
        #context=[location_task, guide_task],
        context=context,
        agent=agent,
        output_file='agentic-ai/crewai/travel_plan.md',
    )


In [11]:

location_task = location_task(
  location_expert,
  from_city,
  destination_city,
  date_from,
  date_to
)

guide_task = guide_task(
  guide_expert,
  destination_city,
  interests,
  date_from,
  date_to
)

planner_task = planner_task(
  [location_task, guide_task],
  planner_expert,
  destination_city,
  interests,
  date_from,
  date_to,
)



In [12]:
from crewai import Crew, Process
crew = Crew(
    agents=[location_expert, guide_expert, planner_expert],
    tasks=[location_task, guide_task, planner_task],
    process=Process.sequential,
    full_output=True,
    share_crew=False,
    verbose=True
)

result = crew.kickoff()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 61b58d1e-d685-4528-a09e-790de9b6a353                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          In French : This task involves a comprehensive data collection process to provide the traveler with    │
│  essential information about their destination. It includes researching and compiling details on various        │
│  accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of       │
│  living in the area. The task also covers transportation options, visa requirements, and any travel advisories  │
│  that may be relevant.                                                                                          │
│          consider also the weather conditions forcast on the travel dates. and all the events that may be       │
│  relevant to the traveler during the trip period.                                                               │
│                                                                                                                 │
│          Traveling from : India                                                                                 │
│          Destination city : Rome                                                                                │
│          Arrival Date : 1st March 2025                                                                          │
│          Departure Date : 7th March 2025                                                                        │
│                                                                                                                 │
│          Follow this rules :                                                                                    │
│          1. if the Rome is in a French country : Respond in FRENCH.                                             │
│                                                                                                                 │
│  ID: 73ee0aa3-67e3-496c-aa24-db7533a8e03a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Trip Expert                                                                                      │
│                                                                                                                 │
│  Task:                                                                                                          │
│          In French : This task involves a comprehensive data collection process to provide the traveler with    │
│  essential information about their destination. It includes researching and compiling details on various        │
│  accommodations, ranging from budget-friendly hostels to luxury hotels, as well as estimating the cost of       │
│  living in the area. The task also covers transportation options, visa requirements, and any travel advisories  │
│  that may be relevant.                                                                                          │
│          consider also the weather conditions forcast on the travel dates. and all the events that may be       │
│  relevant to the traveler during the trip period.                                                               │
│                                                                                                                 │
│          Traveling from : India                                                                                 │
│          Destination city : Rome                                                                                │
│          Arrival Date : 1st March 2025                                                                          │
│          Departure Date : 7th March 2025                                                                        │
│                                                                                                                 │
│          Follow this rules :                                                                                    │
│          1. if the Rome is in a French country : Respond in FRENCH.                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_web_tool                                                                                          │
│  Args: {'query': {'type': 'string'}}                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_web_tool executed with result: Error executing tool: Tool 'search_web_tool' arguments validation failed: 1 validation error for Search_Web_Tool
query
  Input should be a valid string [type=string_type, input_value={'type': 'string'...
Tool search_web_tool executed with result: Error executing tool: Tool 'search_web_tool' arguments validation failed: 1 validation error for Search_Web_Tool
query
  Field required [type=missing, input_value={'single_room': '150EUR p...om': '200...


...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#7) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: search_web_tool                                                                                          │
│  Output: snippet: This guide compiles essential practical information for navigating Rome smoothly in 2026.     │
│  From avoiding common tourist scams to understanding restaurantPublishedFebruary 28, 2026, title: Rome Travel   │
│  Tips 2026: Essential Practical Guide, link:                                                                    │
│  https://www.machupicchu.org/rome-travel-tips-2026-essential-practical-guide.htm, snippet: February 10, 2026    │
│  -Best for: Sophisticated travelers, Vatican sightseers. Where to stay: The First Musica ($$$). Things to do:   │
│  The second-century Castel Sant’Angelo. Where to eat and drink: Gli Esploratori ($$); Pulejo Ristorante ($$$);  │
│  33 Giri ($). Save to list · Once home to the municipal slaughterhouse — now a contemporary art center —        │
│  Testaccio is the birthplace of Rome’s offal-loaded cuisine and today boasts top trattorias and the Mercato di  │
│  Testaccio food market., title: Rome Travel Guide: Where to Eat, Hotels and Places to Visit - The New York      │
│  Times, link: https://www.nytimes.com/interactive/2026/travel/rome-italy-guide.html, snippet: January 1, 2026   │
│  -This comprehensive budget travel guide to Rome with tips on things to do, costs, ways to save money, and      │
│  more!, title: Rome Budget Travel Guide (Updated 2026), link:                                                   │
│  https://www.nomadicmatt.com/travel-guides/italy-travel-tips/rome/, snippet: February 8, 2026 -Explore the      │
│  definitive 2026 Rome travel guide. Discover the Colosseum, Vatican masterpieces, authentic cuisine, and        │
│  hidden layers in the Eternal City., title: The ultimate guide to Rome (2026): what to see, do and know, link:  │
│  https://www.guidetoitaly.com/ultimate-guide-to-rome/, snippet: February 19, 2026 -From the Colosseum and       │
│  Roman Forum to the Vatican and Trevi Fountain, Rome offers unforgettable experiences for every traveler.,      │
│  title: Rome Travel Guide 2026: Best Places, Food, Itinerary & Hidden Gems | YoloPlan, link:                    │
│  https://www.yoloplan.com/articles/rome-travel-guide-2026-best-places-to-visit-food-7-day-itinerary             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Planning Expert                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Rome Travel Guide**                                                                                          │
│  =====================                                                                                          │
│                                                                                                                 │
│  # Welcome to Rome                                                                                              │
│  -----------------                                                                                              │
│                                                                                                                 │
│  ### A Curated Overview of the City                                                                             │
│                                                                                                                 │
│  Rome, the Eternal City, is a treasure trove of history, architecture, art, and delicious cuisine. From         │
│  ancient ruins to modern attractions, Rome has something for every kind of traveler.                            │
│                                                                                                                 │
│  ### What to Expect                                                                                             │
│                                                                                                                 │
│  A 4-phrase markdown-formatted document including:                                                              │
│  - A curated article of presentation of the city,                                                               │
│  - A breakdown of daily living expenses, and spots to visit.                                                    │
│                                                                                                                 │
│  # Daily Living Expenses                                                                                        │
│  ------------------                                                                                             │
│                                                                                                                 │
│  The cost of living in Rome can vary depending on your lifestyle and accommodation choices. Here are some       │
│  estimated daily expenses:                                                                                      │
│                                                                                                                 │
│  *   Food:                                                                                                      │
│      *   Fast food/street food: €5-10 per meal                                                                  │
│      *   Mid-range restaurant: €20-40 per meal                                                                  │
│      *   Fine dining: €50-100 per meal                                                                          │
│  *   Transportation:                                                                                            │
│      *   Metro ticket: €1.50                                                                                    │
│      *   Bus ticket: €1.50                             

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│          This task synthesizes all collected information into a detaileds introduction to the city              │
│  (description of city and presentation, in 3 pragraphes) cohesive and practical travel plan. and takes into     │
│  account the traveler's schedule, preferences, and budget to draft a day-by-day itinerary. The planner also     │
│  provides insights into the city's layout and transportation system to facilitate easy navigation.              │
│          Destination city : Rome                                                                                │
│          interests : sight seeing and good food                                                                 │
│          Arrival Date : 1st March 2025                                                                          │
│          Departure Date : 7th March 2025                                                                        │
│                                                                                                                 │
│          Follow this rules :                                                                                    │
│          1. if the Rome is in a French country : Respond in FRENCH.                                             │
│                                                                                                                 │
│  Agent: Travel Planning Expert                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ d1455ccc-39e3-4c73-b771-26ffd4f3a01c                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/d1455ccc-39e3-4c7 │
│ 3-b771-26ffd4f3a01c?access_code=TRACE-db28ea93e6                             │
│ 🔑 Access Code: TRACE-db28ea93e6                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 61b58d1e-d685-4528-a09e-790de9b6a353                                                                       │
│  Final Output: **Rome Travel Guide**                                                                            │
│  =====================                                                                                          │
│                                                                                                                 │
│  # Welcome to Rome                                                                                              │
│  -----------------                                                                                              │
│                                                                                                                 │
│  ### A Curated Overview of the City                                                                             │
│                                                                                                                 │
│  Rome, the Eternal City, is a treasure trove of history, architecture, art, and delicious cuisine. From         │
│  ancient ruins to modern attractions, Rome has something for every kind of traveler.                            │
│                                                                                                                 │
│  ### What to Expect                                                                                             │
│                                                                                                                 │
│  A 4-phrase markdown-formatted document including:                                                              │
│  - A curated article of presentation of the city,                                                               │
│  - A breakdown of daily living expenses, and spots to visit.                                                    │
│                                                                                                                 │
│  # Daily Living Expenses                                                                                        │
│  ------------------                                                                                             │
│                                                                                                                 │
│  The cost of living in Rome can vary depending on your lifestyle and accommodation choices. Here are some       │
│  estimated daily expenses:                                                                                      │
│                                                                                                                 │
│  *   Food:                                                                                                      │
│      *   Fast food/street food: €5-10 per meal                                                                  │
│      *   Mid-range restaurant: €20-40 per meal                                                                  │
│      *   Fine dining: €50-100 per meal                                                                          │
│  *   Transportation:                                                                                            │
│      *   Metro ticket: €1.50                                                                                    │
│      *   Bus ticket: €1.50                            